# tp-19 追い切りラップの特徴量と調教師別のパターン採点

**問い**: 追い切りのラップ（全体・終い1F・加速/減速）と追い切り日（レース週／1週前／2週前）から、調教師ごとに「どの調教パターンが好走につながるか」を採点した特徴量は、予測を良くするか。

## データ
GCS の `horse_training`（馬別の調教タイム）を `build_training_laps` で 1 本 1 行の表にしたもの（`training_laps.parquet`）。キーは `horse_id`。調教師は `trainer_id`（名前では結合しない）。**調教の日付 < レース日のものだけ**を使う。

## 特徴量
| 群 | 内容 |
|----|------|
| TL_A | 最終（レース週）・1週前・2週前の追い切りについて、種別（坂路/ウッド/ポリ）、所属、強度（馬なり/強め/一杯）、日数、時計の標準化（(種別,所属,強度)ごとに、調教日より前の過去12か月の平均・標準偏差で z 化。小さい = 速い）、終い1F、加速度（終い1F − 直前2Fの1Fあたり平均。負 = 加速）。14日/28日の本数、強い追い切りの本数、馬自身の過去の最終追い切りとの差 |
| TL_B | パターン（最終の種別 × 強度 × 終いの形 × 1週前の強さ × 最終の日数帯）と、**調教師×パターン**の採点。複勝率・勝率を、パターン全体の率へ縮約（Beta-Binomial 型、m=30 / 100 を比較）し、調教師全体の率との差（その厩舎でこの型が効くか）、パターン全体の効き、件数、調教師内の割合（小さい = その厩舎では珍しい型）。成績は**前日までに確定したレース**だけ |
| TL_AB | 両方 |

## 評価
LightGBM（v2 の特徴量セット + 追い切り評価・厩舎コメント + 血統×条件）に足して、valid 2022〜2024 × 2 seed。採用基準 = 勝率 LogLoss の差の 95% 信頼区間の上限 < 0、効果 ≥ 0.0002、連対・複勝が 0.0002 を超えて悪化しない。test 2025 は使わない。
別に、パターンごとの勝率・複勝率・単勝回収率（記述統計。特徴量ではない）を出す。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute
from src.pipeline.features import training_patterns as tp

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OUT = WORK / "training_laps"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP19_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
SMOKE = os.environ.get("TP19_SMOKE") == "1"
mlflow.set_experiment("keiba-tp-training-laps")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
if SMOKE:
    folds = folds[:1]
key = ["race_id", "horse_id"]
if "trainer_id" not in df.columns:
    tt = pd.read_parquet(WORK / "features/train_table.parquet", columns=key + ["trainer_id"]).astype({"race_id": str, "horse_id": str})
    df = df.merge(tt, on=key, how="left")
RD = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
print("行", len(df), "| 調教師 ID の欠損", int(df["trainer_id"].isna().sum()), "| smoke", SMOKE)

## 追い切り表の読み込みと、時計の標準化

In [ ]:
laps = pd.read_parquet(WORK / "features/training_laps.parquet")
print("追い切り", len(laps), "本 | 馬", laps["horse_id"].nunique(), "| 種別", laps["kind"].value_counts().to_dict())
t = tp.add_reference_z(tp.prepare(laps))
print("標準化できた割合（時計あり）:", round(float(t.loc[t['timed'], 't4z'].notna().mean()), 3))
feat = tp.add_self_relative(tp.race_features(t))
M = pd.DataFrame({"horse_id": df["horse_id"].astype(str).to_numpy(), "race_date": RD.to_numpy()}, index=df.index).merge(feat, on=["horse_id", "race_date"], how="left")
M.index = df.index
cov = M["has_training"].notna().groupby(pd.to_datetime(df["date"]).dt.year).mean()
print("レースへの対応率（年別）:", cov.round(3).to_dict())
print("最終追い切り（時計あり）の割合:", round(float(M["f_t1"].notna().mean()), 3), "| 1週前あり:", round(float(M["w1_exists"].fillna(0).mean()), 3))

## 特徴量群を作る

In [ ]:
A_COLS = [c for c in M.columns if c.startswith(("f_", "w1_", "w2_")) and c not in ("f_untimed_later",)] + ["f_untimed_later", "n_work_14", "n_hard_14", "mean_t4z_14", "best_t1z_14", "n_work_28", "slope_share_28", "best_t1z_28", "has_training"]
A_COLS = [c for c in dict.fromkeys(A_COLS) if c in M.columns and c not in ("horse_id", "race_date")]
G = {"TL_A": M[A_COLS].astype("float32")}
races = pd.DataFrame({"date": df["date"].to_numpy(), "trainer_id": df["trainer_id"].astype(str).to_numpy(), "pattern": tp.pattern_key(M).to_numpy(), "finish_position": df["finish_position"].to_numpy()}, index=df.index)
print("パターンの種類", races["pattern"].nunique(), "| パターンありの割合", round(float((races['pattern'] != '').mean()), 3))
for m_ in (30.0, 100.0):
    t0 = time.time()
    S = tp.trainer_pattern_scores(races, m=m_)
    G[f"TL_B_m{int(m_)}"] = S.astype("float32")
    print(f"TL_B m={m_:.0f}: {time.time() - t0:.0f}s 欠損率 {S.isna().mean().mean():.3f}")
G["TL_AB_m30"] = pd.concat([G["TL_A"], G["TL_B_m30"]], axis=1)
G["TL_AB_m100"] = pd.concat([G["TL_A"], G["TL_B_m100"]], axis=1)
for k_, v_ in G.items():
    v_.assign(race_id=df["race_id"].to_numpy(), horse_id=df["horse_id"].to_numpy()).to_parquet(OUT / f"cand_{k_}.parquet", index=False)
print({k_: v_.shape[1] for k_, v_ in G.items()})

## パターンの記述統計（特徴量ではなく、傾向の確認）

In [ ]:
res_all = []
for y in range(2020, 2026):
    res_all.append(pd.read_parquet(REPO_ROOT / f"data/page_reference/tables/{y}/race_result_flat.parquet", columns=["race_id", "horse_id", "odds"]))
OD = pd.concat(res_all).astype({"race_id": str, "horse_id": str}); OD["odds"] = pd.to_numeric(OD["odds"], errors="coerce")
P = pd.concat([races[["pattern", "finish_position"]], df[key]], axis=1).merge(OD, on=key, how="left")
P = P[(P["pattern"] != "") & P["finish_position"].notna()]
P["win"] = (P["finish_position"] == 1).astype(float); P["top3"] = (P["finish_position"] <= 3).astype(float)
P["ret"] = np.where(P["win"] == 1, P["odds"], 0.0)
tab = P.groupby("pattern").agg(n=("win", "size"), 勝率=("win", "mean"), 複勝率=("top3", "mean"), 単勝回収率=("ret", "mean")).query("n >= 1500").sort_values("複勝率", ascending=False)
print("全体: 勝率 %.3f 複勝率 %.3f 単勝回収率 %.3f" % (P["win"].mean(), P["top3"].mean(), P["ret"].mean()))
display(pd.concat([tab.head(10), tab.tail(10)]).round(3))
tab.to_csv(OUT / "pattern_stats.csv")

## 基準との比較（LightGBM、2 seed）

In [ ]:
fin_map = meta.set_index(key)["finish_position"]
res0, OOF0 = zoo_runner.run_oof("lgb_tl_base", "lgb_binary", X0, meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"基準: win {res0['logloss_win'].mean():.5f} top2 {res0['logloss_top2'].mean():.5f} top3 {res0['logloss_top3'].mean():.5f}", flush=True)
ROWS = []
names = list(G) if not SMOKE else ["TL_A"]
for name in names:
    t0 = time.time()
    X = pd.concat([X0, G[name].add_prefix(f"tl_{name}_")], axis=1)
    res, oof = zoo_runner.run_oof(f"lgb_tl_{name}", "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(LP))
    ROWS.append(fb.compare_oof(OOF0, oof, fin_map, name))
    r = ROWS[-1]
    print(f"{name}: win {res['logloss_win'].mean():.5f} Δ {r['Δwin']:+.5f} (上限 {r['win_hi']:+.5f}) top2 {r['Δtop2']:+.5f} top3 {r['Δtop3']:+.5f} 採用={r['採用']} ({time.time() - t0:.0f}s)", flush=True)
    pd.DataFrame(ROWS).set_index("variant").to_csv(OUT / "tl_vs_base.csv")
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))

In [ ]:
import lightgbm as lgb
y = (meta["finish_position"].to_numpy() == 1).astype(int)
bn = R.sort_values("Δwin").index[0]
Xb = pd.concat([X0, G[bn].add_prefix(f"tl_{bn}_")], axis=1)
f = folds[-1]
cats = [c for c in Xb.columns if str(Xb[c].dtype) == "category"]
prm = dict({"objective": "binary", "verbosity": -1, "num_threads": THREADS, "seed": 0, "bagging_freq": 1}, **LP["lgb"])
bst = lgb.train(prm, lgb.Dataset(Xb.iloc[f.train_index], label=y[f.train_index], categorical_feature=cats), num_boost_round=LP["n_rounds"])
g = pd.Series(bst.feature_importance("gain"), index=Xb.columns); g = g / g.sum()
new = [c for c in Xb.columns if c.startswith("tl_")]
print("最良:", bn, "| 新特徴量の gain 合計", round(float(g[new].sum()) * 100, 2), "%"); display(g[new].sort_values(ascending=False).head(12).round(4))
adopted = list(R.index[R["採用"]])
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "coverage_by_year": cov.round(4).to_dict(), "n_patterns": int(races["pattern"].nunique()), "results": R.round(6).reset_index().to_dict("records"),
                                             "adopted": adopted, "best": bn, "gain_share_pct": float(g[new].sum()) * 100}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-19-training-laps", tags={"notebook": "tp-19", "git_commit": GIT_COMMIT}):
    for i_, n in enumerate(R.index):
        mlflow.log_metrics({f"delta_win_g{i_}": float(R.loc[n, "Δwin"])})
    mlflow.log_dict({f"g{i_}": n for i_, n in enumerate(R.index)}, "training_laps/metric_index.json")
    for f_ in ("tl_vs_base.csv", "report.json", "pattern_stats.csv"):
        mlflow.log_artifact(str(OUT / f_), "training_laps")